In [5]:
%pip install torch torchvision torchaudio


   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.5/114.6 MB 3.3 MB/s eta 0:00:35
    --------------------------------------- 1.8/114.6 MB 6.9 MB/s eta 0:00:17
   - -------------------------------------- 3.9/114.6 MB 7.7 MB/s eta 0:00:15
   - -------------------------------------- 5.5/114.6 MB 7.8 MB/s eta 0:00:14
   -- ------------------------------------- 7.6/114.6 MB 8.2 MB/s eta 0:00:14
   --- ------------------------------------ 9.4/114.6 MB 8.3 MB/s eta 0:00:13
   ---- ----------------------------------- 11.5/114.6 MB 8.6 MB/s eta 0:00:12
   ---- ----------------------------------- 13.4/114.6 MB 8.6 MB/s eta 0:00:12
   ----- ---------------------------------- 14.4/114.6 MB 8.6 MB/s eta 0:00:12
   ----- ---------------------------------- 16.8/114.6 MB 8.7 MB/s eta 0:00:12
   ------ --------------------------------- 19.1/114.6 MB 8.8 MB/s eta 0:00:11
   ------- -------------------------------- 21.5/114.6 MB 9.0 MB/

In [6]:
import cv2
import os
import random
import shutil
import numpy as np

import torch
import torch.nn as nn
from torchvision import models, transforms

Resize 400x images to 100x

In [8]:
# First run will be for normal 400x images

# Paths (Update these to match your actual Drive folders)
source_path = r'C:\Users\user\Documents\OSCC Dataset for DATA8009\ftmp4cvtmb-1\Second Set\400x Normal Oral Cavity Histopathological Images'
output_path = r'C:\Users\user\Documents\OSCC Dataset for DATA8009\ftmp4cvtmb-1\First Set\100x Normal Oral Cavity Histopathological Images'

if not os.path.exists(output_path):
    os.makedirs(output_path)

# Scale factor calculation: Target / Current = 100 / 400 = 0.25
scale_factor = 0.25

for img_name in os.listdir(source_path):
    if img_name.endswith(('.png', '.jpg', '.jpeg', '.tif')):
        img = cv2.imread(os.path.join(source_path, img_name))

        if img is not None:
            # Calculate new dimensions
            width = int(img.shape[1] * scale_factor)
            height = int(img.shape[0] * scale_factor)

            # INTER_AREA is the best interpolation method for shrinking images
            rescaled_img = cv2.resize(img, (width, height), interpolation=cv2.INTER_AREA)

            # Save to the new directory
            cv2.imwrite(os.path.join(output_path, img_name), rescaled_img)

print("Processing complete.")


Processing complete.


In [9]:
# Second run will be for cancer 400x images

# Paths (Update these to match your actual Drive folders)
source_path = r'C:\Users\user\Documents\OSCC Dataset for DATA8009\ftmp4cvtmb-1\Second Set\400x OSCC Histopathological Images'
output_path = r'C:\Users\user\Documents\OSCC Dataset for DATA8009\ftmp4cvtmb-1\First Set\100x OSCC Histopathological Images'

if not os.path.exists(output_path):
    os.makedirs(output_path)

# Scale factor calculation: Target / Current = 100 / 400 = 0.25
scale_factor = 0.25

for img_name in os.listdir(source_path):
    if img_name.endswith(('.png', '.jpg', '.jpeg', '.tif')):
        img = cv2.imread(os.path.join(source_path, img_name))

        if img is not None:
            # Calculate new dimensions
            width = int(img.shape[1] * scale_factor)
            height = int(img.shape[0] * scale_factor)

            # INTER_AREA is the best interpolation method for shrinking images
            rescaled_img = cv2.resize(img, (width, height), interpolation=cv2.INTER_AREA)

            # Save to the new directory
            cv2.imwrite(os.path.join(output_path, img_name), rescaled_img)

print("Processing complete.")


Processing complete.


Train/Val/Test split

In [10]:
# Create a function to split the data into train, val, and test sets

def split_data(source, train_dir, val_dir, test_dir, split_ratio=(0.7, 0.2, 0.1)):
    # 1. Get list of all files
    files = os.listdir(source)
    random.shuffle(files)

    # 2. Calculate split indices
    train_count = int(len(files) * split_ratio[0])
    val_count = int(len(files) * split_ratio[1])

    # 3. Slice the list
    train_files = files[:train_count]
    val_files = files[train_count:train_count + val_count]
    test_files = files[train_count + val_count:]

    # 4. Move files
    for f in train_files:
        shutil.copy(os.path.join(source, f), os.path.join(train_dir, f))
    for f in val_files:
        shutil.copy(os.path.join(source, f), os.path.join(val_dir, f))
    for f in test_files:
        shutil.copy(os.path.join(source, f), os.path.join(test_dir, f))

In [11]:
# Now assign values to the variabled for the function

# Source and directories for (CANCER at 100x)
source_oscc = r'C:\Users\user\Documents\OSCC Dataset for DATA8009\ftmp4cvtmb-1\First Set\100x OSCC Histopathological Images'
train_dir_oscc = r'C:\Users\user\Documents\OSCC Dataset for DATA8009\Sorted oscc\Train\Cancer'
val_dir_oscc = r'C:\Users\user\Documents\OSCC Dataset for DATA8009\Sorted oscc\Val\Cancer'
test_dir_oscc = r'C:\Users\user\Documents\OSCC Dataset for DATA8009\Sorted oscc\Test\Cancer'

In [12]:
split_data(source_oscc, train_dir_oscc, val_dir_oscc, test_dir_oscc)

In [13]:
# Source and directories for NORMAL (at 100x)
source_normal = r'C:\Users\user\Documents\OSCC Dataset for DATA8009\ftmp4cvtmb-1\First Set\100x Normal Oral Cavity Histopathological Images'
train_dir_normal = r'C:\Users\user\Documents\OSCC Dataset for DATA8009\Sorted oscc\Train\Normal'
val_dir_normal = r'C:\Users\user\Documents\OSCC Dataset for DATA8009\Sorted oscc\Val\Normal'
test_dir_normal = r'C:\Users\user\Documents\OSCC Dataset for DATA8009\Sorted oscc\Test\Normal'

In [14]:
split_data(source_normal, train_dir_normal, val_dir_normal, test_dir_normal)

Preprocessing Stage

Create patches (224x224)

In [15]:
# Define a function for patch extraction. These patches will be saved to a new location (obviously permanently).

def extract_patches(source_dir, output_dir, patch_size=224):
    if not os.path.exists(output_dir): os.makedirs(output_dir)

    for img_name in os.listdir(source_dir):
        img = cv2.imread(os.path.join(source_dir, img_name))
        if img is None: continue

        h, w, _ = img.shape
        # Iterate through the image in steps of 224
        for i in range(0, h - patch_size + 1, patch_size):
            for j in range(0, w - patch_size + 1, patch_size):
                patch = img[i:i+patch_size, j:j+patch_size]

                # Check for "empty" (mostly white) patches
                # If mean brightness is > 230, it's likely background
                if np.mean(patch) < 230:
                    patch_name = f"{os.path.splitext(img_name)[0]}_p_{i}_{j}.png"
                    cv2.imwrite(os.path.join(output_dir, patch_name), patch)



In [16]:
# Use function
# First for normal images (train)
extract_patches(r'C:\Users\user\Documents\OSCC Dataset for DATA8009\Sorted oscc\Train\Normal',
                r'C:\Users\user\Documents\OSCC Dataset for DATA8009\Patches of sorted\Train\Normal')


In [17]:
# Next for oscc images (train)
extract_patches(r'C:\Users\user\Documents\OSCC Dataset for DATA8009\Sorted oscc\Train\Cancer',
                r'C:\Users\user\Documents\OSCC Dataset for DATA8009\Patches of sorted\Train\Cancer')

Using finetuned ResNet18 as a Feature Extractor

In ResNet architectures, the "feature extractor" is usually everything up until the final fully connected (fc) layer. When fine-tuning, we let the gradients flow through these layers so the model can adapt its pre-trained ImageNet knowledge to the specific textures of histopathology.


In [18]:

# 1. Load ResNet18 with pre-trained weights
resnet18 = models.resnet18(weights='IMAGENET1K_V1')

# 2. Fine-tuning: Ensure all parameters are "unfrozen"
# By setting requires_grad = True, the model will update its weights
# for your cancer images during training.
for param in resnet18.parameters():
    param.requires_grad = True

# 3. Create the Feature Extractor
# We remove the final classification layer (resnet18.fc)
# 'list(resnet18.children())[:-1]' takes everything except the last layer
modules = list(resnet18.children())[:-1]
feature_extractor = nn.Sequential(*modules, nn.Flatten())

# Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
feature_extractor = feature_extractor.to(device)

# ResNet18 output size is 512 (unlike AlexNet's 9216)
print("Feature extractor ready. Output dimension: 512")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\user/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:06<00:00, 7.26MB/s]


Feature extractor ready. Output dimension: 512


 Creating the New Classification Network (ANN)

Now, we create a simple Multi-Layer Perceptron (MLP) that takes those 9216 features and learns to classify them as "Normal" or "Cancer."

In [19]:
class CancerClassifier(nn.Module):
    def __init__(self):
        super(CancerClassifier, self).__init__()
        # Input is 512 for ResNet18
        self.mlp = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.mlp(x)

model = CancerClassifier().to(device)


Data Loading (? and transformation)

In [21]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Define transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load data
patch_dataset = datasets.ImageFolder(root=r'C:\Users\user\Documents\OSCC Dataset for DATA8009\Patches of sorted\Train', transform=transform)
dataloader = DataLoader(patch_dataset, batch_size=32, shuffle=True)


Setup Optimizer and Loss --
We use Binary Cross Entropy because this is a cancer vs. normal classification task.

In [22]:
import torch.optim as optim

# Combine parameters from the feature extractor AND the custom ANN
params_to_update = list(feature_extractor.parameters()) + list(model.parameters())

# Use a small learning rate for fine-tuning (1e-5 is a safe start)
optimizer = optim.Adam(params_to_update, lr=0.00001)

# Loss function remains the same for binary classification
criterion = nn.BCELoss()

# Number of epochs
epochs = 10


The Training Loop (Fine-Tuning Mode)

In [24]:
epochs = 10

for epoch in range(epochs):
    # Set both parts to training mode
    feature_extractor.train()
    model.train()

    running_loss = 0.0
    correct_preds = 0
    total_samples = 0

    for inputs, labels in dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device).float().unsqueeze(1)

        optimizer.zero_grad() # Clear gradients for both parts

        # --- Step 1: Feature Extraction (Gradients ARE calculated here now) ---
        features = feature_extractor(inputs)

        # --- Step 2: Classification ---
        outputs = model(features)
        loss = criterion(outputs, labels)

        # --- Step 3: Backpropagation (Updates both ResNet and ANN) ---
        loss.backward()
        optimizer.step()

        # Stats
        running_loss += loss.item() * inputs.size(0)
        predictions = (outputs > 0.5).float()
        correct_preds += (predictions == labels).sum().item()
        total_samples += labels.size(0)

    epoch_loss = running_loss / total_samples
    epoch_acc = correct_preds / total_samples

    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.4f}")

print("Fine-tuning complete!")


Epoch [1/10] - Loss: 0.2750 - Acc: 0.8726
Epoch [2/10] - Loss: 0.1435 - Acc: 0.9458
Epoch [3/10] - Loss: 0.0834 - Acc: 0.9703
Epoch [4/10] - Loss: 0.0548 - Acc: 0.9815
Epoch [5/10] - Loss: 0.0428 - Acc: 0.9850
Epoch [6/10] - Loss: 0.0316 - Acc: 0.9895
Epoch [7/10] - Loss: 0.0285 - Acc: 0.9910
Epoch [8/10] - Loss: 0.0256 - Acc: 0.9910
Epoch [9/10] - Loss: 0.0241 - Acc: 0.9918
Epoch [10/10] - Loss: 0.0237 - Acc: 0.9915
Fine-tuning complete!


Save model for later

In [25]:
# Define the file name and path
save_path = "oscc_detection_ResnetAnnFT.pth"

# Create a dictionary to hold everything
checkpoint = {
    'feature_extractor_state': feature_extractor.state_dict(),
    'classifier_state': model.state_dict(),
    'optimizer_state': optimizer.state_dict(),
    'class_to_idx': patch_dataset.class_to_idx  # Helpful for later inference
}

# Save to disk
torch.save(checkpoint, save_path)
print(f"Model saved successfully to {save_path}!")


Model saved successfully to oscc_detection_ResnetAnnFT.pth!


#### Loading the models
First re-create the model architectures (the same classes you defined earlier) and then load the saved weights into them

In [ ]:
# 1. Re-initialize your models (architecture must be exactly the same)
# Make sure your CancerClassifier and feature_extractor definitions are run first
feature_extractor = ... # (Your ResNet18 code from earlier)
model = CancerClassifier()

# 2. Load the saved checkpoint
checkpoint = torch.load("cancer_detection_model.pth")

# 3. Apply the weights to the models
feature_extractor.load_state_dict(checkpoint['feature_extractor_state'])
model.load_state_dict(checkpoint['classifier_state'])

# 4. Move to GPU if available and set to EVALUATION mode
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
feature_extractor.to(device).eval()
model.to(device).eval()

print("Models loaded and ready for inference!")


#### Model Testing/Validation